<a href="https://colab.research.google.com/github/imtisalrangrez/DeepLearning_Lab/blob/main/DL_week9(180).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------------------------------------
# 1. Load CIFAR-10 and Pick a Sample Image
# ---------------------------------------------------------
(x_train, y_train), (_, _) = tf.keras.datasets.cifar10.load_data()

class_names = ['Airplane','Automobile','Bird','Cat','Deer',
               'Dog','Frog','Horse','Ship','Truck']

# Pick the 6th Automobile (label 1) — rich color/edge features make maps more interesting
# y_train is shape (50000, 1) in CIFAR-10, so flatten first
y_train_flat = y_train.flatten()
auto_idx = np.where(y_train_flat == 1)[0][5]

# Normalize — no reshape needed, already (32, 32, 3)
image = x_train[auto_idx:auto_idx+1] / 255.0          # shape: (1, 32, 32, 3)

# ---------------------------------------------------------
# 2. Build and Quickly Train a Simple CNN
# ---------------------------------------------------------
model = tf.keras.Sequential([
    tf.keras.Input(shape=(32, 32, 3)),
    tf.keras.layers.Conv2D(16, (3, 3), padding='same', activation='relu', name='my_conv'),
    tf.keras.layers.MaxPooling2D((2, 2), name='my_pool'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 1 epoch — just enough for filters to learn basic edges and color gradients
print("Quickly training model to initialize filters...")
model.fit(
    x_train / 255.0,
    y_train_flat,
    epochs=1,
    batch_size=256,
    verbose=0
)

# ---------------------------------------------------------
# 3. Create the Feature Extractor Model
# ---------------------------------------------------------
# Outputs activations from both Conv and Pool layers directly
extractor = tf.keras.models.Model(
    inputs=model.inputs,
    outputs=[
        model.get_layer('my_conv').output,
        model.get_layer('my_pool').output
    ]
)

# Pass the single automobile image through
conv_features, pool_features = extractor.predict(image)

print(f"Conv layer output shape  : {conv_features.shape}")   # (1, 32, 32, 16)
print(f"Pool layer output shape  : {pool_features.shape}")   # (1, 16, 16, 16)

# ---------------------------------------------------------
# 4. Visualization Function
# ---------------------------------------------------------
def plot_feature_maps(feature_maps, title, num_filters=16):
    """Plots a grid of single-channel feature map activations."""
    plt.figure(figsize=(20, 3))
    plt.suptitle(title, fontsize=14)
    for i in range(num_filters):
        plt.subplot(1, num_filters, i + 1)
        # feature_maps shape: (1, Height, Width, Filters)
        # Each slice [:, :, i] is a single learned activation map
        plt.imshow(feature_maps[0, :, :, i], cmap='viridis')
        plt.title(f'F{i+1}', fontsize=8)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# 5. Display Original Image
# ---------------------------------------------------------
plt.figure(figsize=(3, 3))
plt.imshow(image[0])                                   # RGB — no cmap needed
plt.title(f"Original: {class_names[1]} (Label 1)", fontsize=12)
plt.axis('off')
plt.show()

# ---------------------------------------------------------
# 6. Display Feature Maps
# ---------------------------------------------------------
plot_feature_maps(
    conv_features,
    "Conv Layer Output — Edge & Color Gradient Detection (32×32 → 32×32)"
)

plot_feature_maps(
    pool_features,
    "MaxPool Layer Output — Downsampled Dominant Features (32×32 → 16×16)"
)

# ---------------------------------------------------------
# What to observe in CIFAR-10 feature maps vs Fashion MNIST:
# ---------------------------------------------------------
# Fashion MNIST (grayscale):
#   → Filters detect simple edges and silhouette outlines
#   → Feature maps are clean with clear bright/dark contrast
#
# CIFAR-10 (RGB color):
#   → Filters detect color gradients, texture transitions, and directional edges
#   → Some filters respond to specific color channels (reds, blues, greens)
#   → Feature maps look noisier — real-world images have far more visual complexity
#   → After MaxPooling, dominant activations become clearer as noise is suppressed